# VinDr-Mammo Stratified Downloader v2

Downloads **~1000 stratified DICOM files** from PhysioNet VinDr-Mammo dataset:
- **~250 malignant** (BI-RADS 4, 5, 6)
- **~750 benign** (BI-RADS 1, 2)
- **Excludes BI-RADS 3**
- **Patient-level selection** (all images from selected patients)
- **Controlled total file count** (prevents downloading 3000+ files)

## Requirements
1. Google Colab environment
2. PhysioNet account with VinDr-Mammo access
3. Google Drive mounted

## Changes from v1:
- **Fixed:** Now samples at patient-level to control total file count
- **Fixed:** Prevents expansion from 1000 to 3500+ files
- **Added:** Iterative patient sampling to stay close to target

---

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from typing import List, Dict, Tuple
import getpass
import pandas as pd
import numpy as np
from tqdm import tqdm
import time
from datetime import datetime
from collections import defaultdict

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted successfully!")

## Step 2: Stratified Downloader Class (v2 - Fixed)

In [ ]:
class VinDrMammoStratifiedDownloader:
    """
    Stratified downloader for VinDr-Mammo dataset (v2 - with controlled file count).
    
    Downloads ~1000 images with stratification:
    - ~250 malignant (BI-RADS 4, 5, 6)
    - ~750 benign (BI-RADS 1, 2)
    - Excludes BI-RADS 3
    - Patient-level sampling to control total files
    """
    
    def __init__(self, gdrive_path='/content/drive/MyDrive/vindr-mammo-stratified', max_total_files=1000):
        """
        Initialize stratified downloader.
        
        Args:
            gdrive_path: Google Drive path for output
            max_total_files: Maximum total files to download (default: 1000)
        """
        self.base_dir = Path(gdrive_path)
        self.base_url = "https://physionet.org/files/vindr-mammo/1.0.0"
        
        self.username = None
        self.password = None
        
        # Stratification parameters
        self.max_total_files = max_total_files
        self.target_malignant_ratio = 0.25  # 25% malignant
        self.target_benign_ratio = 0.75     # 75% benign
        self.random_seed = 42
        
        # Create directories
        self.base_dir.mkdir(parents=True, exist_ok=True)
        (self.base_dir / 'images').mkdir(exist_ok=True)
        (self.base_dir / 'metadata').mkdir(exist_ok=True)
        (self.base_dir / 'logs').mkdir(exist_ok=True)
        
        self.progress_file = self.base_dir / 'download_progress.json'
        self.selection_file = self.base_dir / 'metadata' / 'selected_files.csv'
        
        print(f"✅ Initialized downloader at {self.base_dir}")
        print(f"   Target: ~{max_total_files} files ({int(max_total_files * 0.25)} malignant, {int(max_total_files * 0.75)} benign)")
    
    def setup_credentials(self, username: str = None, password: str = None) -> bool:
        """Setup PhysioNet credentials."""
        if not username:
            print("\n🔐 PhysioNet Credentials Required")
            print("   Get credentials at: https://physionet.org/")
            username = input("Username: ").strip()
            password = getpass.getpass("Password: ")
        
        self.username = username
        self.password = password
        
        print("🔍 Verifying credentials...")
        return self._test_access()
    
    def _test_access(self) -> bool:
        """Test PhysioNet access."""
        test_url = f"{self.base_url}/SHA256SUMS.txt"
        test_file = self.base_dir / "test_access.txt"
        
        cmd = [
            'wget',
            f'--user={self.username}',
            f'--password={self.password}',
            '-O', str(test_file),
            '-q', '--tries=2', '--timeout=15',
            test_url
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, timeout=20)
            
            if result.returncode == 0 and test_file.exists():
                test_file.unlink()
                print("✅ Credentials verified!")
                return True
            else:
                print("❌ Authentication failed!")
                return False
        except Exception as e:
            print(f"❌ Error: {e}")
            return False
    
    def download_metadata(self) -> bool:
        """Download metadata CSV files."""
        print("\n📊 Downloading Metadata Files")
        print("=" * 70)
        
        metadata_dir = self.base_dir / 'metadata'
        csv_file = 'breast-level_annotations.csv'
        
        url = f"{self.base_url}/{csv_file}"
        output_file = metadata_dir / csv_file
        
        if output_file.exists():
            print(f"  ✅ {csv_file} already exists")
            return True
        
        print(f"  📥 Downloading {csv_file}...", end=" ", flush=True)
        
        cmd = [
            'wget',
            f'--user={self.username}',
            f'--password={self.password}',
            '-O', str(output_file),
            '-q', '--timeout=60', '--tries=3',
            url
        ]
        
        try:
            result = subprocess.run(cmd, capture_output=True, timeout=90)
            
            if result.returncode == 0 and output_file.exists():
                size_mb = output_file.stat().st_size / (1024 * 1024)
                print(f"✅ ({size_mb:.2f} MB)")
                return True
            else:
                print("❌ Failed")
                return False
        except Exception as e:
            print(f"❌ Error: {e}")
            return False
    
    def perform_stratified_selection(self) -> pd.DataFrame:
        """
        Perform patient-level stratified selection with controlled file count.
        
        Returns:
            DataFrame with selected images
        """
        print("\n🎯 Performing Stratified Selection (Patient-Level)")
        print(f"   Target: ~{self.max_total_files} total files")
        print("=" * 70)
        
        # Load metadata
        csv_file = self.base_dir / 'metadata' / 'breast-level_annotations.csv'
        
        if not csv_file.exists():
            print("❌ Metadata not found. Run download_metadata() first.")
            return None
        
        df = pd.read_csv(csv_file)
        print(f"  📊 Total images in dataset: {len(df)}")
        
        # Extract numeric BI-RADS values
        df['birads_numeric'] = df['breast_birads'].str.extract(r'(\d+)')[0].astype(float)
        
        # Exclude BI-RADS 3
        df_filtered = df[df['birads_numeric'] != 3].copy()
        print(f"  ✅ After excluding BI-RADS 3: {len(df_filtered)} images")
        
        # Classify as malignant or benign
        df_filtered['label'] = df_filtered['birads_numeric'].apply(
            lambda x: 1 if x in [4, 5, 6] else 0
        )
        
        malignant_df = df_filtered[df_filtered['label'] == 1]
        benign_df = df_filtered[df_filtered['label'] == 0]
        
        print(f"\n  📊 Classification:")
        print(f"     Malignant (BI-RADS 4, 5, 6): {len(malignant_df)} images")
        print(f"     Benign (BI-RADS 1, 2): {len(benign_df)} images")
        
        # Group by patient
        print(f"\n  🔍 Patient-level analysis...")
        malignant_patient_sizes = malignant_df.groupby('study_id').size().reset_index(name='image_count')
        benign_patient_sizes = benign_df.groupby('study_id').size().reset_index(name='image_count')
        
        print(f"     Malignant patients: {len(malignant_patient_sizes)}")
        print(f"     Benign patients: {len(benign_patient_sizes)}")
        print(f"     Avg images/malignant patient: {malignant_patient_sizes['image_count'].mean():.1f}")
        print(f"     Avg images/benign patient: {benign_patient_sizes['image_count'].mean():.1f}")
        
        # Iterative patient sampling
        print(f"\n  🎲 Iterative patient sampling (seed={self.random_seed})...")
        np.random.seed(self.random_seed)
        
        target_malignant_files = int(self.max_total_files * self.target_malignant_ratio)
        target_benign_files = int(self.max_total_files * self.target_benign_ratio)
        
        # Shuffle patients
        malignant_patient_sizes = malignant_patient_sizes.sample(frac=1, random_state=self.random_seed).reset_index(drop=True)
        benign_patient_sizes = benign_patient_sizes.sample(frac=1, random_state=self.random_seed).reset_index(drop=True)
        
        # Select malignant patients
        selected_malignant_patients = []
        malignant_file_count = 0
        
        for _, row in malignant_patient_sizes.iterrows():
            if malignant_file_count >= target_malignant_files:
                break
            selected_malignant_patients.append(row['study_id'])
            malignant_file_count += row['image_count']
        
        # Select benign patients
        selected_benign_patients = []
        benign_file_count = 0
        
        for _, row in benign_patient_sizes.iterrows():
            if benign_file_count >= target_benign_files:
                break
            selected_benign_patients.append(row['study_id'])
            benign_file_count += row['image_count']
        
        # Get all images from selected patients
        malignant_selected = malignant_df[malignant_df['study_id'].isin(selected_malignant_patients)]
        benign_selected = benign_df[benign_df['study_id'].isin(selected_benign_patients)]
        
        # Combine
        selected_df = pd.concat([malignant_selected, benign_selected], ignore_index=True)
        
        print(f"\n  ✅ Selection Complete:")
        print(f"     Malignant patients: {len(selected_malignant_patients)}")
        print(f"     Malignant images: {len(malignant_selected)}")
        print(f"     Benign patients: {len(selected_benign_patients)}")
        print(f"     Benign images: {len(benign_selected)}")
        print(f"     Total patients: {len(selected_malignant_patients) + len(selected_benign_patients)}")
        print(f"     Total images: {len(selected_df)} (target was {self.max_total_files})")
        
        # Show view distribution
        if 'laterality' in selected_df.columns and 'view_position' in selected_df.columns:
            view_counts = selected_df.groupby(['laterality', 'view_position']).size()
            print(f"\n  📊 View Distribution:")
            for (lat, view), count in view_counts.items():
                print(f"     {lat} {view}: {count} images")
        
        # Save selection
        selected_df.to_csv(self.selection_file, index=False)
        print(f"\n  💾 Selection saved to: {self.selection_file}")
        
        return selected_df
    
    def download_selected_files(self, selected_df: pd.DataFrame = None) -> bool:
        """Download selected files."""
        print("\n📥 Downloading Selected Files")
        print("=" * 70)
        
        if selected_df is None:
            if not self.selection_file.exists():
                print("❌ No selection file found. Run perform_stratified_selection() first.")
                return False
            selected_df = pd.read_csv(self.selection_file)
        
        print(f"  📊 Files to download: {len(selected_df)}")
        
        # Load progress
        progress = self._load_progress()
        downloaded_files = set(progress.get('downloaded_files', []))
        
        success_count = 0
        fail_count = 0
        skip_count = 0
        
        # Download each file
        for idx, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Downloading"):
            study_id = row['study_id']
            image_id = row['image_id']
            
            image_path = f"images/{study_id}/{image_id}.dicom"
            
            if image_path in downloaded_files:
                skip_count += 1
                success_count += 1
                continue
            
            url = f"{self.base_url}/{image_path}"
            output_file = self.base_dir / image_path
            output_file.parent.mkdir(parents=True, exist_ok=True)
            
            if self._download_file(url, output_file):
                success_count += 1
                downloaded_files.add(image_path)
                progress['downloaded_files'].append(image_path)
            else:
                fail_count += 1
                progress.setdefault('failed_files', []).append(image_path)
            
            if (success_count + fail_count) % 50 == 0:
                self._save_progress(progress)
        
        self._save_progress(progress)
        
        print(f"\n{'=' * 70}")
        print(f"✅ Download Complete!")
        print(f"   Downloaded: {success_count - skip_count} new files")
        print(f"   Skipped (already exist): {skip_count}")
        print(f"   Failed: {fail_count}")
        print(f"   Total files: {len(progress['downloaded_files'])}")
        print(f"{'=' * 70}\n")
        
        return fail_count == 0
    
    def _download_file(self, url: str, output_file: Path, max_retries: int = 3) -> bool:
        """Download single file with retry."""
        for attempt in range(max_retries):
            cmd = [
                'wget',
                f'--user={self.username}',
                f'--password={self.password}',
                '-O', str(output_file),
                '-c', '-q',
                '--timeout=60',
                '--tries=2',
                url
            ]
            
            try:
                result = subprocess.run(cmd, capture_output=True, timeout=90)
                
                if result.returncode == 0 and output_file.exists() and output_file.stat().st_size > 0:
                    return True
                
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
            
            except Exception:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
        
        return False
    
    def _load_progress(self) -> Dict:
        """Load download progress."""
        if self.progress_file.exists():
            try:
                with open(self.progress_file, 'r') as f:
                    return json.load(f)
            except:
                pass
        return {'downloaded_files': [], 'failed_files': []}
    
    def _save_progress(self, progress: Dict):
        """Save download progress."""
        progress['last_update'] = datetime.now().isoformat()
        with open(self.progress_file, 'w') as f:
            json.dump(progress, f, indent=2)
    
    def get_status(self):
        """Display download status."""
        print("\n" + "=" * 70)
        print("📊 DOWNLOAD STATUS")
        print("=" * 70)
        
        if self.selection_file.exists():
            selected_df = pd.read_csv(self.selection_file)
            print(f"📋 Selected files: {len(selected_df)}")
            
            if 'label' in selected_df.columns:
                malignant = (selected_df['label'] == 1).sum()
                benign = (selected_df['label'] == 0).sum()
                print(f"   Malignant: {malignant}")
                print(f"   Benign: {benign}")
        else:
            print("📋 No selection file found")
        
        progress = self._load_progress()
        print(f"\n📥 Download progress:")
        print(f"   Downloaded: {len(progress['downloaded_files'])}")
        print(f"   Failed: {len(progress.get('failed_files', []))}")
        
        print("=" * 70 + "\n")

## Step 3: Initialize Downloader

**Set max_total_files parameter to control the total number of files**

In [ ]:
# Initialize downloader with controlled file count
downloader = VinDrMammoStratifiedDownloader(
    gdrive_path='/content/drive/MyDrive/vindr-mammo-stratified',
    max_total_files=1000  # Change this to control total files
)

## Step 4: Setup PhysioNet Credentials

In [ ]:
# Setup credentials (will prompt for username and password)
downloader.setup_credentials()

# Or provide credentials directly:
# downloader.setup_credentials(username='your_username', password='your_password')

## Step 5: Download Metadata

In [ ]:
# Download metadata CSV
downloader.download_metadata()

## Step 6: Perform Stratified Selection

**This version uses patient-level sampling to control the total file count**

In [ ]:
# Perform stratified selection
selected_df = downloader.perform_stratified_selection()

# Show first few selected files
print("\n📋 First 10 selected files:")
display(selected_df[['study_id', 'image_id', 'breast_birads', 'label']].head(10))

## Step 7: Download Selected Files

**Note:** This will download the selected DICOM files to Google Drive.

In [ ]:
# Download selected files
downloader.download_selected_files(selected_df)

## Step 8: Check Status

In [ ]:
# Check download status
downloader.get_status()

## Summary

This notebook (v2) fixes the file count explosion issue:

### What was fixed:
- **v1 Problem:** Sampled 1000 images, then expanded to ALL images from those patients → 3514 files
- **v2 Solution:** Sample patients iteratively until reaching ~1000 total files

### Result:
- **~250 malignant images** from selected malignant patients
- **~750 benign images** from selected benign patients
- **~1000 total files** (controlled by max_total_files parameter)
- **Excludes BI-RADS 3**
- **Random seed = 42** for reproducibility

---